# Xem thông tin audio hàng loạt

Notebook này dùng để xem thông tin nhiều file audio cùng lúc.

Bạn chỉ cần điền mảng `audio_paths`, notebook sẽ gọi `ffprobe` và hiển thị:

- file có tồn tại không
- thời lượng
- sample rate
- số kênh
- codec
- bitrate
- dung lượng file
- định dạng/container

## 1. Import và hàm hỗ trợ

Cần có `ffprobe` trong PATH. Nếu máy đã chạy được RVC/FFmpeg thì thường đã ổn.

In [1]:
import json
import subprocess
from pathlib import Path
from IPython.display import Audio, display


def seconds_to_hms(seconds):
    if seconds is None:
        return ""
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    if h:
        return f"{h:02d}:{m:02d}:{s:06.3f}"
    return f"{m:02d}:{s:06.3f}"


def run_ffprobe(path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        str(path),
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        return None, proc.stderr.strip() or proc.stdout.strip()
    return json.loads(proc.stdout), ""


def probe_audio(path_value):
    path = Path(path_value).expanduser()
    row = {
        "path": str(path),
        "file_name": path.name,
        "exists": path.is_file(),
        "duration_sec": None,
        "duration": "",
        "sample_rate": None,
        "channels": None,
        "channel_layout": "",
        "codec": "",
        "bit_rate": None,
        "format": "",
        "size_mb": None,
        "error": "",
    }

    if not row["exists"]:
        row["error"] = "Không tìm thấy file"
        return row

    row["size_mb"] = round(path.stat().st_size / (1024 * 1024), 3)
    data, error = run_ffprobe(path)
    if error:
        row["error"] = error
        return row

    audio_streams = [s for s in data.get("streams", []) if s.get("codec_type") == "audio"]
    if not audio_streams:
        row["error"] = "Không tìm thấy audio stream"
        return row

    stream = audio_streams[0]
    fmt = data.get("format", {})
    duration = fmt.get("duration") or stream.get("duration")
    bit_rate = fmt.get("bit_rate") or stream.get("bit_rate")

    row.update({
        "duration_sec": round(float(duration), 3) if duration else None,
        "duration": seconds_to_hms(duration) if duration else "",
        "sample_rate": int(stream["sample_rate"]) if stream.get("sample_rate") else None,
        "channels": int(stream["channels"]) if stream.get("channels") else None,
        "channel_layout": stream.get("channel_layout", ""),
        "codec": stream.get("codec_name", ""),
        "bit_rate": int(bit_rate) if bit_rate else None,
        "format": fmt.get("format_name", ""),
    })
    return row


## 2. Nhập danh sách đường dẫn audio

Sửa mảng bên dưới thành các file bạn muốn kiểm tra.

In [4]:
audio_paths = [
    r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\song.mp3",
    r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\output\song_(vocals)_vocals_mel_band_roformer.mp3",
    r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\output\song_(other)_vocals_mel_band_roformer.mp3",
    r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\output\song_(Instrumental)_model_bs_roformer_ep_317_sdr_12.mp3",
    r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\output\song_(Vocals)_model_bs_roformer_ep_317_sdr_12.mp3"
]


## 3. Chạy probe và hiển thị bảng thông tin

In [5]:
rows = [probe_audio(path) for path in audio_paths]

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    display_df = df.rename(columns={
        "path": "đường_dẫn",
        "file_name": "tên_file",
        "exists": "tồn_tại",
        "duration_sec": "thời_lượng_giây",
        "duration": "thời_lượng",
        "sample_rate": "sample_rate",
        "channels": "số_kênh",
        "channel_layout": "bố_cục_kênh",
        "codec": "codec",
        "bit_rate": "bitrate",
        "format": "định_dạng",
        "size_mb": "dung_lượng_mb",
        "error": "lỗi",
    })
    display(display_df)
except ImportError:
    for row in rows:
        print(json.dumps(row, indent=2, ensure_ascii=False))


{
  "path": "D:\\DUT_ITF\\Semester_10th\\do_an_tot_nghiep\\project\\audio_separator_application\\python-audio-separator\\song.mp3",
  "file_name": "song.mp3",
  "exists": true,
  "duration_sec": 244.062,
  "duration": "04:04.062",
  "sample_rate": 44100,
  "channels": 2,
  "channel_layout": "stereo",
  "codec": "mp3",
  "bit_rate": 127384,
  "format": "mp3",
  "size_mb": 3.706,
  "error": ""
}
{
  "path": "D:\\DUT_ITF\\Semester_10th\\do_an_tot_nghiep\\project\\audio_separator_application\\python-audio-separator\\output\\song_(vocals)_vocals_mel_band_roformer.mp3",
  "file_name": "song_(vocals)_vocals_mel_band_roformer.mp3",
  "exists": true,
  "duration_sec": 244.062,
  "duration": "04:04.062",
  "sample_rate": 44100,
  "channels": 2,
  "channel_layout": "stereo",
  "codec": "mp3",
  "bit_rate": 320035,
  "format": "mp3",
  "size_mb": 9.311,
  "error": ""
}
{
  "path": "D:\\DUT_ITF\\Semester_10th\\do_an_tot_nghiep\\project\\audio_separator_application\\python-audio-separator\\output\\s

## 4. Lọc các file lỗi hoặc khác sample rate

Cell này hữu ích để kiểm tra dataset training hoặc file trước khi mix.

In [4]:
valid_rows = [row for row in rows if row["exists"] and not row["error"]]
error_rows = [row for row in rows if row["error"]]

print("Tổng số file:", len(rows))
print("File hợp lệ:", len(valid_rows))
print("File lỗi:", len(error_rows))

if error_rows:
    print("\nCác file có lỗi:")
    for row in error_rows:
        print("-", row["path"], "=>", row["error"])

sample_rates = sorted({row["sample_rate"] for row in valid_rows if row["sample_rate"]})
channels = sorted({row["channels"] for row in valid_rows if row["channels"]})
codecs = sorted({row["codec"] for row in valid_rows if row["codec"]})

print("\nCác sample rate:", sample_rates)
print("Các số kênh:", channels)
print("Các codec:", codecs)


Tổng số file: 1
File hợp lệ: 1
File lỗi: 0

Các sample rate: [44100]
Các số kênh: [2]
Các codec: ['mp3']


## 5. Nghe thử một file bất kỳ

Đổi `preview_index` để nghe file trong danh sách.

In [5]:
preview_index = 0

preview_path = Path(audio_paths[preview_index])
if preview_path.is_file():
    display(Audio(str(preview_path)))
else:
    print("Không tìm thấy file:", preview_path)


## 6. Xuất CSV nếu cần

Nếu `pandas` đã cài, cell này ghi bảng thông tin ra `audio_info_report.csv`.

In [ ]:
csv_path = Path("audio_info_report.csv")

try:
    import pandas as pd

    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("Đã lưu:", csv_path.resolve())
except ImportError:
    print("Chưa cài pandas; bỏ qua bước xuất CSV")
